# E-Commerce Sales & Customer Behavior Analysis
### UCI Online Retail Dataset | Python EDA | Google Colab

**Goal:** Explore sales performance, product demand, country-level revenue, customer purchasing behavior, cancellations, outliers, time trends, and RFM customer segments.

**Dataset source:** [UCI Machine Learning Repository — Online Retail](https://archive.ics.uci.edu/dataset/352/online+retail)

The original dataset covers transactions from **1 December 2010 to 9 December 2011** and includes invoice, product, quantity, date, price, customer, and country fields.

### How to use this notebook
1. Open this notebook in Google Colab.
2. Run cells from top to bottom.
3. When prompted, upload `Online Retail.xlsx` downloaded from the UCI page.
4. Review the charts and tables generated from your actual data.
5. Replace the business-recommendation template at the end with conclusions supported by your outputs.

**Important:** This notebook does not hard-code analytical results. All reported metrics are calculated from the uploaded dataset.

## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_columns", 50)

print("Libraries imported successfully.")

## 2. Load the Excel dataset
Upload the original `Online Retail.xlsx` file when prompted.

In [ ]:
from google.colab import files

uploaded = files.upload()
if not uploaded:
    raise ValueError("No file uploaded. Please upload Online Retail.xlsx.")

file_name = next(iter(uploaded))
df_original = pd.read_excel(file_name)

print(f"Loaded file: {file_name}")
print(f"Rows: {df_original.shape[0]:,} | Columns: {df_original.shape[1]}")
display(df_original.head())

## 3. Understand the data

In [ ]:
print("Column names:")
print(df_original.columns.tolist())

print("\nData types and non-null counts:")
df_original.info()

print("\nDescriptive statistics:")
display(df_original.describe(include="all").T)

print("\nFirst and last transaction dates (raw values):")
print(df_original["InvoiceDate"].min(), "to", df_original["InvoiceDate"].max())

## 4. Missing-value and duplicate analysis

In [ ]:
missing_summary = pd.DataFrame({
    "Missing_Count": df_original.isna().sum(),
    "Missing_Percent": (df_original.isna().mean() * 100).round(2)
}).sort_values("Missing_Count", ascending=False)

display(missing_summary)

duplicate_count = df_original.duplicated().sum()
print(f"Exact duplicate rows: {duplicate_count:,}")

missing_nonzero = missing_summary[missing_summary["Missing_Count"] > 0]
if not missing_nonzero.empty:
    ax = missing_nonzero["Missing_Count"].sort_values().plot(
        kind="barh", figsize=(9, 5), title="Missing Values by Column"
    )
    ax.set_xlabel("Number of missing values")
    ax.set_ylabel("Column")
    plt.tight_layout()
    plt.show()
else:
    print("No missing values detected.")

## 5. Data cleaning and feature engineering
We retain an untouched copy (`df_original`) and create a working copy. Exact duplicates are removed. Missing customer IDs are not filled with invented IDs. Cancellation invoices are flagged for separate analysis.

In [ ]:
df = df_original.copy()
df = df.drop_duplicates().copy()

# Normalize important data types
df["InvoiceNo"] = df["InvoiceNo"].astype("string").str.strip()
df["StockCode"] = df["StockCode"].astype("string").str.strip()
df["Description"] = df["Description"].astype("string").str.strip()
df["Country"] = df["Country"].astype("string").str.strip()
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["UnitPrice"] = pd.to_numeric(df["UnitPrice"], errors="coerce")

# Cancellation flag: UCI documentation says invoice numbers beginning with C indicate cancellation.
df["IsCancelled"] = df["InvoiceNo"].str.upper().str.startswith("C", na=False)

# Line-level amount; negative quantities remain negative and are not silently removed here.
df["Revenue"] = df["Quantity"] * df["UnitPrice"]

print(f"Rows in original data: {len(df_original):,}")
print(f"Rows after exact duplicate removal: {len(df):,}")
print(f"Cancelled invoice lines: {df['IsCancelled'].sum():,}")
print(f"Rows with missing CustomerID: {df['CustomerID'].isna().sum():,}")
display(df.head())

## 6. Define the regular-sales analysis population
For product, revenue, and RFM analysis, this notebook uses rows that are not flagged as cancelled, have valid invoice dates, positive quantities and positive unit prices. This is a transparent analysis rule, not a claim that every excluded row is erroneous. Returns/cancellations are analyzed separately below.

In [ ]:
valid_base = (
    df["InvoiceDate"].notna()
    & df["InvoiceNo"].notna()
    & df["StockCode"].notna()
    & df["Description"].notna()
    & df["Quantity"].notna()
    & df["UnitPrice"].notna()
)

sales = df.loc[
    valid_base
    & (~df["IsCancelled"])
    & (df["Quantity"] > 0)
    & (df["UnitPrice"] > 0)
].copy()

sales["Revenue"] = sales["Quantity"] * sales["UnitPrice"]
sales["YearMonth"] = sales["InvoiceDate"].dt.to_period("M").astype(str)
sales["Weekday"] = sales["InvoiceDate"].dt.day_name()
sales["Hour"] = sales["InvoiceDate"].dt.hour
sales["Date"] = sales["InvoiceDate"].dt.date

print(f"Rows in regular-sales analysis: {len(sales):,}")
print(f"Unique invoices: {sales['InvoiceNo'].nunique():,}")
print(f"Unique products (StockCode): {sales['StockCode'].nunique():,}")
print(f"Customers with IDs: {sales['CustomerID'].nunique():,}")
print(f"Gross line revenue in this filtered population: £{sales['Revenue'].sum():,.2f}")

## 7. KPI summary

In [ ]:
order_revenue = sales.groupby("InvoiceNo")["Revenue"].sum()
kpis = pd.Series({
    "Regular sales rows": len(sales),
    "Unique invoices/orders": sales["InvoiceNo"].nunique(),
    "Unique products": sales["StockCode"].nunique(),
    "Identified customers": sales["CustomerID"].nunique(),
    "Total quantity sold": sales["Quantity"].sum(),
    "Gross revenue (£)": sales["Revenue"].sum(),
    "Average order value (£)": order_revenue.mean(),
    "Median order value (£)": order_revenue.median()
}, name="Value")

display(kpis.to_frame())

## 8. Univariate analysis: quantities and prices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(sales["Quantity"], bins=50, ax=axes[0])
axes[0].set_title("Distribution of Quantity per Line")
axes[0].set_xlabel("Quantity")
axes[0].set_xlim(0, sales["Quantity"].quantile(0.99))

sns.histplot(sales["UnitPrice"], bins=50, ax=axes[1])
axes[1].set_title("Distribution of Unit Price")
axes[1].set_xlabel("Unit price (£)")
axes[1].set_xlim(0, sales["UnitPrice"].quantile(0.99))

plt.tight_layout()
plt.show()

## 9. Top 10 products by revenue — horizontal bar chart

In [ ]:
top_products_revenue = (
    sales.groupby("Description")["Revenue"]
    .sum()
    .nlargest(10)
    .sort_values()
)

display(top_products_revenue.sort_values(ascending=False).to_frame("Revenue (£)"))

top_products_revenue.plot(kind="barh", figsize=(10, 6))
plt.title("Top 10 Products by Revenue")
plt.xlabel("Revenue (£)")
plt.ylabel("Product")
plt.tight_layout()
plt.show()

## 10. Top 10 products by quantity sold

In [ ]:
top_products_quantity = (
    sales.groupby("Description")["Quantity"]
    .sum()
    .nlargest(10)
    .sort_values()
)

display(top_products_quantity.sort_values(ascending=False).to_frame("Quantity sold"))

top_products_quantity.plot(kind="barh", figsize=(10, 6))
plt.title("Top 10 Products by Quantity Sold")
plt.xlabel("Total quantity")
plt.ylabel("Product")
plt.tight_layout()
plt.show()

## 11. Revenue by country — bar chart

In [ ]:
country_revenue = sales.groupby("Country")["Revenue"].sum().sort_values(ascending=False)
display(country_revenue.head(10).to_frame("Revenue (£)"))

country_revenue.head(10).sort_values().plot(kind="barh", figsize=(10, 6))
plt.title("Top 10 Countries by Revenue")
plt.xlabel("Revenue (£)")
plt.ylabel("Country")
plt.tight_layout()
plt.show()

## 12. Revenue contribution by country — pie chart
The pie chart shows the top five countries separately and combines all remaining countries into `Others` so it remains readable.

In [ ]:
top5_countries = country_revenue.head(5)
others_revenue = country_revenue.iloc[5:].sum()
pie_values = pd.concat([top5_countries, pd.Series({"Others": others_revenue})])

plt.figure(figsize=(8, 8))
plt.pie(
    pie_values.values,
    labels=pie_values.index,
    autopct="%1.1f%%",
    startangle=90
)
plt.title("Share of Regular-Sales Revenue by Country")
plt.tight_layout()
plt.show()

## 13. Monthly revenue trend — line chart

In [ ]:
monthly_revenue = (
    sales.set_index("InvoiceDate")["Revenue"]
    .resample("MS")
    .sum()
)

display(monthly_revenue.to_frame("Revenue (£)"))

monthly_revenue.plot(marker="o", figsize=(12, 5))
plt.title("Monthly Revenue Trend")
plt.xlabel("Month")
plt.ylabel("Revenue (£)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("Note: the first and last months may be partial because the dataset covers a limited date range.")

## 14. Orders by weekday

In [ ]:
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekday_orders = (
    sales.groupby("Weekday")["InvoiceNo"]
    .nunique()
    .reindex(weekday_order, fill_value=0)
)

display(weekday_orders.to_frame("Unique invoices"))

weekday_orders.plot(kind="bar", figsize=(10, 5))
plt.title("Number of Orders by Weekday")
plt.xlabel("Weekday")
plt.ylabel("Unique invoices")
plt.xticks(rotation=35)
plt.tight_layout()
plt.show()

## 15. Hour-of-day pattern

In [ ]:
hour_orders = sales.groupby("Hour")["InvoiceNo"].nunique().reindex(range(24), fill_value=0)

hour_orders.plot(kind="bar", figsize=(11, 5))
plt.title("Unique Invoices by Hour of Day")
plt.xlabel("Hour (24-hour clock)")
plt.ylabel("Unique invoices")
plt.tight_layout()
plt.show()

## 16. Average order value
Average order value (AOV) is the mean of invoice-level revenue in the regular-sales population. The median is also shown because order values can be skewed.

In [ ]:
order_summary = sales.groupby("InvoiceNo").agg(
    OrderRevenue=("Revenue", "sum"),
    OrderQuantity=("Quantity", "sum"),
    OrderDate=("InvoiceDate", "min")
)

print(f"Number of invoices: {len(order_summary):,}")
print(f"Average order value: £{order_summary['OrderRevenue'].mean():,.2f}")
print(f"Median order value: £{order_summary['OrderRevenue'].median():,.2f}")

sns.histplot(order_summary["OrderRevenue"], bins=50)
plt.title("Distribution of Invoice-Level Order Revenue")
plt.xlabel("Order revenue (£)")
plt.xlim(0, order_summary["OrderRevenue"].quantile(0.99))
plt.tight_layout()
plt.show()

## 17. Customer value: top 10 identified customers

In [ ]:
customer_revenue = (
    sales.dropna(subset=["CustomerID"])
    .groupby("CustomerID")["Revenue"]
    .sum()
    .sort_values(ascending=False)
)

display(customer_revenue.head(10).to_frame("Revenue (£)"))

customer_revenue.head(10).sort_values().plot(kind="barh", figsize=(10, 6))
plt.title("Top 10 Customers by Revenue")
plt.xlabel("Revenue (£)")
plt.ylabel("Customer ID")
plt.tight_layout()
plt.show()

## 18. Outlier analysis with box plots
Box plots help identify extreme values; an outlier is not automatically an error. Whiskers are shown on the central 98% display range for readability, without deleting those rows from `sales`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(x=sales["Quantity"], ax=axes[0])
axes[0].set_title("Quantity per Line — Box Plot")
axes[0].set_xlim(0, sales["Quantity"].quantile(0.99))

sns.boxplot(x=sales["UnitPrice"], ax=axes[1])
axes[1].set_title("Unit Price — Box Plot")
axes[1].set_xlim(0, sales["UnitPrice"].quantile(0.99))

plt.tight_layout()
plt.show()

print("Quantity quantiles:")
display(sales["Quantity"].quantile([0.5, 0.9, 0.95, 0.99, 0.999]).to_frame("Quantity"))
print("Unit price quantiles:")
display(sales["UnitPrice"].quantile([0.5, 0.9, 0.95, 0.99, 0.999]).to_frame("UnitPrice (£)"))

## 19. Price versus quantity — scatter plot and correlation
A sample is used for plotting performance. Pearson correlation describes a linear association; it does not prove that price causes quantity changes.

In [ ]:
sample = sales.sample(n=min(5000, len(sales)), random_state=42)

sns.scatterplot(data=sample, x="UnitPrice", y="Quantity", alpha=0.35)
plt.title("Unit Price vs Quantity Purchased (Random Sample)")
plt.xlabel("Unit price (£)")
plt.ylabel("Quantity")
plt.xlim(0, sample["UnitPrice"].quantile(0.99))
plt.ylim(0, sample["Quantity"].quantile(0.99))
plt.tight_layout()
plt.show()

pearson_corr = sales["UnitPrice"].corr(sales["Quantity"], method="pearson")
spearman_corr = sales["UnitPrice"].corr(sales["Quantity"], method="spearman")
print(f"Pearson correlation: {pearson_corr:.3f}")
print(f"Spearman correlation: {spearman_corr:.3f}")

## 20. Correlation heatmap

In [ ]:
numeric_columns = ["Quantity", "UnitPrice", "Revenue"]
corr = sales[numeric_columns].corr()

plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1, fmt=".2f")
plt.title("Correlation Heatmap: Quantity, Unit Price and Revenue")
plt.tight_layout()
plt.show()

## 21. Cancellation and return analysis
Cancellation invoice IDs begin with `C`. Negative quantities can also indicate returns/credit adjustments. We calculate cancellation rates at both line and invoice levels because they answer different questions.

In [ ]:
# Cancellation line rate among rows with invoice numbers
invoice_rows = df[df["InvoiceNo"].notna()].copy()
cancelled_line_rate = invoice_rows["IsCancelled"].mean() * 100

# Unique invoices and cancelled invoice IDs
invoice_flags = (
    invoice_rows.groupby("InvoiceNo")["IsCancelled"]
    .any()
)
cancelled_invoice_rate = invoice_flags.mean() * 100

negative_quantity_rows = (df["Quantity"] < 0).sum()
negative_quantity_rate = (df["Quantity"] < 0).mean() * 100

print(f"Cancelled line percentage: {cancelled_line_rate:.2f}%")
print(f"Cancelled invoice percentage: {cancelled_invoice_rate:.2f}%")
print(f"Rows with negative quantity: {negative_quantity_rows:,} ({negative_quantity_rate:.2f}%)")

cancel_chart = pd.Series({
    "Non-cancelled lines": int((~invoice_rows["IsCancelled"]).sum()),
    "Cancelled lines": int(invoice_rows["IsCancelled"].sum())
})
cancel_chart.plot(kind="pie", autopct="%1.1f%%", figsize=(7, 7), ylabel="")
plt.title("Cancelled vs Non-cancelled Invoice Lines")
plt.tight_layout()
plt.show()

# Examine negative quantity rows separately
display(df.loc[df["Quantity"] < 0, ["InvoiceNo", "StockCode", "Description", "Quantity", "UnitPrice", "InvoiceDate"]].head(10))

## 22. RFM customer segmentation
RFM means **Recency, Frequency, Monetary value**:
- **Recency:** days since the customer's most recent purchase (lower is better).
- **Frequency:** number of distinct invoices (higher is generally better).
- **Monetary:** total revenue in the regular-sales population (higher is generally better).

The reference date is set to one day after the latest invoice timestamp in the filtered sales dataset, making the result reproducible for this dataset. Customers without a CustomerID cannot be assigned to individual RFM segments.

In [ ]:
rfm_source = sales.dropna(subset=["CustomerID"]).copy()

if rfm_source.empty:
    raise ValueError("No identified customers are available for RFM analysis.")

reference_date = rfm_source["InvoiceDate"].max().normalize() + pd.Timedelta(days=1)

rfm = rfm_source.groupby("CustomerID").agg(
    LastPurchase=("InvoiceDate", "max"),
    Frequency=("InvoiceNo", "nunique"),
    Monetary=("Revenue", "sum")
)

rfm["Recency"] = (reference_date - rfm["LastPurchase"].dt.normalize()).dt.days
rfm = rfm[["Recency", "Frequency", "Monetary"]]

print("RFM reference date:", reference_date.date())
display(rfm.head())
display(rfm.describe().T)

## 23. RFM scoring
Each metric is scored from 1 to 5 using quintiles. For Recency, lower values receive higher scores; for Frequency and Monetary, higher values receive higher scores. Duplicate quantile boundaries can occur, so the code uses rank-based quintiles to remain robust.

In [ ]:
# Rank first to handle repeated values and avoid duplicate qcut edges.
rfm["R_Score"] = pd.qcut(
    rfm["Recency"].rank(method="first", ascending=True),
    5, labels=[5, 4, 3, 2, 1]
).astype(int)

rfm["F_Score"] = pd.qcut(
    rfm["Frequency"].rank(method="first", ascending=True),
    5, labels=[1, 2, 3, 4, 5]
).astype(int)

rfm["M_Score"] = pd.qcut(
    rfm["Monetary"].rank(method="first", ascending=True),
    5, labels=[1, 2, 3, 4, 5]
).astype(int)

rfm["RFM_Score"] = (
    rfm["R_Score"].astype(str)
    + rfm["F_Score"].astype(str)
    + rfm["M_Score"].astype(str)
)

def assign_segment(row):
    r, f, m = row["R_Score"], row["F_Score"], row["M_Score"]
    if r >= 4 and f >= 4 and m >= 4:
        return "Best Customers"
    elif r >= 3 and f >= 3:
        return "Loyal / Active"
    elif r >= 4 and f <= 2:
        return "Recent Customers"
    elif r <= 2 and f >= 3:
        return "At-Risk Valuable"
    elif r <= 2 and f <= 2:
        return "Hibernating / Low Activity"
    else:
        return "Potential / Developing"

rfm["Segment"] = rfm.apply(assign_segment, axis=1)

display(rfm.sort_values("Monetary", ascending=False).head(15))

## 24. RFM segment charts and summary

In [ ]:
segment_counts = rfm["Segment"].value_counts()
display(segment_counts.to_frame("Customers"))

segment_counts.plot(kind="bar", figsize=(11, 5))
plt.title("Number of Customers by RFM Segment")
plt.xlabel("Segment")
plt.ylabel("Number of customers")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

segment_revenue = rfm.groupby("Segment")["Monetary"].sum().sort_values(ascending=False)
display(segment_revenue.to_frame("Monetary revenue (£)"))

segment_revenue.sort_values().plot(kind="barh", figsize=(10, 5))
plt.title("Revenue by RFM Segment")
plt.xlabel("Monetary value (£)")
plt.ylabel("Segment")
plt.tight_layout()
plt.show()

## 25. Product demand trend: compare first and second halves of observed months
This is a simple exploratory comparison, not a forecast. Only products appearing in both periods are compared, and periods are based on the available dataset range.

In [ ]:
monthly_product_qty = (
    sales.assign(MonthStart=sales["InvoiceDate"].dt.to_period("M").dt.to_timestamp())
    .groupby(["MonthStart", "Description"])["Quantity"]
    .sum()
    .reset_index()
)

if not monthly_product_qty.empty:
    months = sorted(monthly_product_qty["MonthStart"].unique())
    midpoint = len(months) // 2
    first_months = months[:midpoint]
    second_months = months[midpoint:]

    first_qty = (
        monthly_product_qty[monthly_product_qty["MonthStart"].isin(first_months)]
        .groupby("Description")["Quantity"].sum()
    )
    second_qty = (
        monthly_product_qty[monthly_product_qty["MonthStart"].isin(second_months)]
        .groupby("Description")["Quantity"].sum()
    )

    demand_compare = pd.concat(
        [first_qty.rename("First_Period_Quantity"),
         second_qty.rename("Second_Period_Quantity")],
        axis=1
    ).fillna(0)

    demand_compare = demand_compare[
        (demand_compare["First_Period_Quantity"] > 0)
        & (demand_compare["Second_Period_Quantity"] > 0)
    ]

    demand_compare["Change"] = (
        demand_compare["Second_Period_Quantity"]
        - demand_compare["First_Period_Quantity"]
    )
    demand_compare = demand_compare.sort_values("Change", ascending=False)

    print("Largest increases (among products present in both periods):")
    display(demand_compare.head(10))

    print("Largest decreases (among products present in both periods):")
    display(demand_compare.tail(10).sort_values("Change"))
else:
    print("No product-month data available.")

## 26. Export analysis tables
These CSV files can be downloaded from Colab and used in a report or dashboard.

In [ ]:
from google.colab import files

sales.head(0).to_csv("sales_analysis_schema.csv", index=False)
rfm.to_csv("rfm_customer_segments.csv")
country_revenue.to_csv("country_revenue.csv", header=["Revenue"])
top_products_revenue.to_csv("top_products_revenue.csv", header=["Revenue"])

print("Exported: rfm_customer_segments.csv, country_revenue.csv, top_products_revenue.csv")
files.download("rfm_customer_segments.csv")

## 27. Business insights and recommendations — fill from your actual results

Use the tables and charts above to write evidence-based conclusions. Do **not** claim a result until you verify it in your notebook.

1. **Revenue drivers:** Name the leading product and its revenue from Section 9. Consider monitoring availability for high-revenue items.
2. **Geographical performance:** Name the highest-revenue country from Section 11. Consider tailored campaigns for markets with meaningful revenue and growth potential.
3. **Seasonality:** Describe the strongest and weakest complete months in Section 13. Note that boundary months may be partial.
4. **Customer retention:** Use the RFM segment chart in Section 24. Consider loyalty offers for Best Customers and reactivation campaigns for At-Risk Valuable customers.
5. **Cancellations and returns:** Report the line-level and invoice-level rates from Section 21 separately. Investigate operational causes before proposing changes.
6. **Demand planning:** Use Sections 10 and 25 to compare high-volume products and changing demand. Avoid treating one period comparison as a reliable forecast.
7. **Pricing:** Interpret the correlations in Sections 19–20 as associations only. Use controlled experiments or deeper analysis before claiming a causal price effect.

### Final conclusion template
> This project analyzed the UCI Online Retail transactions using Python, Pandas, NumPy, Matplotlib and Seaborn. The analysis examined data quality, sales and country performance, product demand, order values, time trends, cancellations, outliers, correlations, and RFM customer segments. The key findings were **[insert verified findings from your outputs]**. Based on these results, the recommended actions are **[insert specific recommendations supported by evidence]**.

### Resume project title
**E-Commerce Sales & Customer Behavior Analysis using Python**

### Resume bullet template
- Performed exploratory data analysis on the UCI Online Retail dataset using Python, Pandas, NumPy, Matplotlib and Seaborn.
- Analyzed revenue trends, product demand, country performance, order value, cancellations and transaction outliers.
- Built RFM customer segments using recency, invoice frequency and monetary value to support retention-focused recommendations.
- Presented data-backed findings through charts, summary tables and business recommendations.